# v4.6 — NASDAQ Divergence Signal

| | |
|---|---|
| **Version** | v4.6 |
| **Key change** | Extra filter: only enter if NASDAQ also confirms bearish direction |
| **Data source** | 2024 real 1-min NSE option CSVs + ^IXIC daily from yfinance |
| **Lookahead** | None — D-1 NASDAQ close |
| **Baseline** | v2/backtest |

**Why**: v2 uses only S&P 500 for the US signal. NASDAQ is tech-heavy and often leads
broad market direction. On days where NASDAQ diverges from S&P (e.g., S&P up but
NASDAQ flat), the bearish fade signal may be less reliable.

**Change**: After bearish combo fires, additionally require `nasdaq_ret < NASDAQ_DIV_THRESH`.
Default: `NASDAQ_DIV_THRESH = 0.0` (NASDAQ must be non-positive on trade days).
This is a filter on top of existing v2 combos — it reduces trade count but may improve WR.


In [1]:
VERSION    = 'v4.6'
KEY_CHANGE = 'NASDAQ confirmation filter: only trade if NASDAQ also non-positive (D-1)'
DATA_SRC   = 'real_2024_options'

SL_PCT    = 0.15
TP_PCT    = 0.40
BASE_LOTS = 2
MAX_LOTS  = 10
DTE0_LOTS = 5
LOT_SIZE  = 75
BEAR_N    = 10
BASE_RATE = 54.5
ENTRY_TIME      = '09:25'
EXIT_TIME       = '11:15'
INITIAL_CAPITAL = 3000.0   # fixed starting capital for all versions (Rs)

NASDAQ_CONFIRM     = True   # require NASDAQ confirmation
NASDAQ_DIV_THRESH  = 0.0    # NASDAQ D-1 return must be < this to confirm bearish


In [2]:
import sys, os
from pathlib import Path

HERE    = Path(globals().get('__vsc_ipynb_file__', Path.cwd())).parent
V4_DIR  = HERE.parent if HERE.name.startswith('v4') else HERE.parent.parent / 'v4'
sys.path.insert(0, str(V4_DIR))

from _engine import (simulate_trades, compute_metrics, save_results,
                     sig_map, combo_fires, load_reliable_signals, print_summary)

import pandas as pd
import numpy  as np
import yfinance as yf
import pickle, warnings
from datetime import date, timedelta, time as dtime
from pathlib import Path
warnings.filterwarnings('ignore')


In [3]:
# ── Paths ──────────────────────────────────────────────────────────────────────
# HERE.parent.parent == gap_trading/ for both v2/backtest and v4/v4.x layouts
GAP_TRADING     = HERE.parent.parent
DATA_2024       = GAP_TRADING / 'v2' / 'backtesting_2024_options' / '2024'
SIGNALS_CSV     = GAP_TRADING / 'v2' / 'v2_reliable_signals.csv'
NIFTY_SPOT_DIR  = DATA_2024 / '2024Nifty'
EXPIRY_CSV      = DATA_2024 / 'expiry.csv'
CACHE_FILE      = HERE / 'results' / f'trade_cache_{VERSION.replace(".", "_")}.pkl'
# ── NSE holidays and event days (2024) ────────────────────────────────────────
NSE_HOLIDAYS = {
    date(2024,  1, 22), date(2024,  3, 25), date(2024,  3, 29),
    date(2024,  4, 14), date(2024,  4, 17), date(2024,  5,  1),
    date(2024,  6, 17), date(2024,  7, 17), date(2024,  8, 15),
    date(2024, 10,  2), date(2024, 11, 15), date(2024, 12, 25),
}
EVENT_DAYS = {
    date(2024,  2,  1),   # Union Budget
    date(2024,  2,  8),   # RBI MPC
    date(2024,  4,  5),   # RBI MPC
    date(2024,  6,  7),   # RBI MPC
    date(2024,  8,  8),   # RBI MPC
    date(2024, 10,  9),   # RBI MPC
    date(2024, 12,  6),   # RBI MPC
}
# ── Expiry helpers ─────────────────────────────────────────────────────────────
expiry_df = pd.read_csv(EXPIRY_CSV)
expiry_df.columns = [c.strip() for c in expiry_df.columns]
# Column is 'ExpiryDate' in DDMMMYY format (e.g. '04JAN24')
_ecol = [c for c in expiry_df.columns if 'expiry' in c.lower() or 'date' in c.lower()][0]
expiry_dates = sorted(
    pd.to_datetime(expiry_df[_ecol].str.strip(), format='%d%b%y').dt.date.tolist()
)

def nearest_expiry(d):
    for e in expiry_dates:
        if e >= d:
            return e
    return expiry_dates[-1]

def expiry_folder(exp):
    return '2024' + exp.strftime('%b').capitalize()
# ── Global market data (yfinance, daily) ──────────────────────────────────────
print("Fetching global market data ...", end=' ', flush=True)
START = '2023-12-15'
END   = '2025-01-10'
TICKERS = {'SP500': '^GSPC', 'SGX': 'NKD=F', 'DAX': '^GDAXI',
           'VIX': '^VIX', 'NIFTY': '^NSEI'}

raw = {}
for name, ticker in TICKERS.items():
    try:
        df = yf.download(ticker, start=START, end=END, progress=False, auto_adjust=True)
        if isinstance(df.columns, pd.MultiIndex):
            df.columns = df.columns.get_level_values(0)
        if df.index.tz is None:
            df.index = df.index.tz_localize('UTC')
        raw[name] = df[['Close']].rename(columns={'Close': name})
    except Exception as e:
        print(f"  [warn] {name}: {e}")
        raw[name] = pd.DataFrame()

gdf = pd.concat([v for v in raw.values() if not v.empty], axis=1)
gdf.index = gdf.index.date
gdf = gdf.ffill()
print(f"done. {len(gdf)} rows.")
# ── NASDAQ daily data ──────────────────────────────────────────────────────────
print("Fetching NASDAQ ...", end=' ', flush=True)
try:
    nq_df = yf.download('^IXIC', start='2023-12-01', end='2025-01-15',
                         progress=False, auto_adjust=True)
    if isinstance(nq_df.columns, pd.MultiIndex):
        nq_df.columns = nq_df.columns.get_level_values(0)
    if nq_df.index.tz is None:
        nq_df.index = nq_df.index.tz_localize('UTC')
    nq_df.index = nq_df.index.date
    nq_close = nq_df['Close'].to_dict()
    print(f"done. {len(nq_close)} days.")
except Exception as e:
    nq_close = {}
    print(f"FAILED ({e}) — NASDAQ filter will not apply.")
# ── Load signal combos ────────────────────────────────────────────────────────
bear_combos = load_reliable_signals(SIGNALS_CSV, bear_n=BEAR_N, base_rate=BASE_RATE)
print(f"Loaded {len(bear_combos)} bearish combos from {SIGNALS_CSV.name}")
for i, c in enumerate(bear_combos, 1):
    print(f"  {i:>2}. {c}")
# ── NIFTY spot 1-min data ─────────────────────────────────────────────────────
spot_cache = {}
for f in sorted(NIFTY_SPOT_DIR.glob('Nifty-2024*.csv')):
    try:
        s = pd.read_csv(f)
        s.columns = [c.strip().lower() for c in s.columns]
        s['datetime'] = pd.to_datetime(s['datetime'])
        s['date'] = s['datetime'].dt.date
        s['time'] = s['datetime'].dt.time
        for d, grp in s.groupby('date'):
            spot_cache[d] = grp.reset_index(drop=True)
    except Exception as e:
        print(f"  [warn] spot {f.name}: {e}")
print(f"Spot data loaded for {len(spot_cache)} days.")
# ── Build signal_days dict ────────────────────────────────────────────────────
if CACHE_FILE.exists():
    with open(CACHE_FILE, 'rb') as f:
        signal_days = pickle.load(f)
    print(f"Loaded trade cache: {len(signal_days)} dates, "
          f"{sum(1 for v in signal_days.values() if v is not None)} with signal.")
else:
    signal_days = {}
    skipped = []
    LOT_SIZE_LOCAL = 75
    STRIKE_STEP    = 50

    all_dates = sorted([d for d in gdf.index if isinstance(d, date)
                        and date(2024,1,1) <= d <= date(2024,12,31)])

    for d in all_dates:
        # ── Skip conditions ────────────────────────────────────────────────
        if d.weekday() == 0:           # Monday
            skipped.append((d, 'Monday')); continue
        if d in NSE_HOLIDAYS:
            skipped.append((d, 'Holiday')); continue
        if d in EVENT_DAYS:
            skipped.append((d, 'EventDay')); continue

        # ── Global data row ────────────────────────────────────────────────
        prev_rows = [x for x in gdf.index if x < d]
        if len(prev_rows) < 2:
            skipped.append((d, 'NoGlobalData')); continue
        prev  = prev_rows[-1]
        prev2 = prev_rows[-2]

        def _ret(col, d1, d2):
            try:
                return float((gdf.loc[d1, col] - gdf.loc[d2, col]) / gdf.loc[d2, col])
            except Exception:
                return None

        sp500_ret = _ret('SP500', prev, prev2)
        sgx_ret   = _ret('SGX',   prev, prev2)
        dax_ret   = _ret('DAX',   prev, prev2)
        vix_ret   = _ret('VIX',   prev, prev2)
        pind_ret  = _ret('NIFTY', prev, prev2)

        # ── NIFTY open (9:15) for gap ──────────────────────────────────────
        spot = spot_cache.get(d)
        if spot is None:
            skipped.append((d, 'NoSpotData')); continue
        open_row = spot[spot['time'] == dtime(9, 15)]
        if open_row.empty:
            skipped.append((d, 'No915Open')); continue
        nifty_open = float(open_row.iloc[0]['open'])

        nifty_prev_close = float(gdf.loc[prev, 'NIFTY']) if prev in gdf.index else None
        if nifty_prev_close is None or nifty_prev_close <= 0:
            skipped.append((d, 'NoPrevClose')); continue

        gap = (nifty_open - nifty_prev_close) / nifty_prev_close

        # ── Compute signals ────────────────────────────────────────────────
        sigs = sig_map(gap, pind_ret, sp500_ret, sgx_ret, dax_ret, vix_ret)
        fired = [c for c in bear_combos if combo_fires(c, sigs)]

        # ── v4.6: NASDAQ confirmation filter ──────────────────────────────────────
        if NASDAQ_CONFIRM and fired:
            prev_rows2 = [x for x in gdf.index if x < d]
            if len(prev_rows2) >= 2:
                p, p2 = prev_rows2[-1], prev_rows2[-2]
                nq_today  = nq_close.get(p)
                nq_yest   = nq_close.get(p2)
                if nq_today is not None and nq_yest is not None and nq_yest > 0:
                    nasdaq_ret = (nq_today - nq_yest) / nq_yest
                    if nasdaq_ret >= NASDAQ_DIV_THRESH:
                        skipped.append((d, 'NASDAQ_not_confirm'))
                        signal_days[d] = None
                        continue

        if not fired:
            signal_days[d] = None; continue

        winning_combo = fired[0]

        # ── NIFTY 9:25 for strike ──────────────────────────────────────────
        row_925 = spot[spot['time'] == dtime(9, 25)]
        if row_925.empty:
            signal_days[d] = None; continue
        nifty_925 = float(row_925.iloc[0]['open'])
        atm    = round(nifty_925 / STRIKE_STEP) * STRIKE_STEP
        strike = atm - STRIKE_STEP   # 1-OTM put

        # ── Expiry ─────────────────────────────────────────────────────────
        exp = nearest_expiry(d)
        dte = (exp - d).days
        exp_str = exp.strftime('%d%b%y').upper()

        # ── Load option file ───────────────────────────────────────────────
        month_dir = DATA_2024 / expiry_folder(exp)
        opt_file  = month_dir / f"NIFTY-{exp_str}-{d.strftime('%d%b%y').upper()}.csv"
        if not opt_file.exists():
            signal_days[d] = None; continue

        try:
            opt = pd.read_csv(opt_file)
            opt.columns = [c.strip().lower() for c in opt.columns]
            opt['datetime'] = pd.to_datetime(opt['datetime'])
            opt['time']     = opt['datetime'].dt.time
            pe = opt[(opt['strike_price'] == strike) & (opt['right'].str.strip().str.upper() == 'PE')]
            if pe.empty:
                signal_days[d] = None; continue

            # Entry candle at 9:25
            entry_row = pe[pe['time'] == dtime(9, 25)]
            if entry_row.empty:
                entry_row = pe[pe['time'] >= dtime(9, 25)].head(1)
            if entry_row.empty:
                signal_days[d] = None; continue
            entry_price = float(entry_row.iloc[0]['open'])
            if entry_price <= 0:
                signal_days[d] = None; continue

            # Candles from 9:25 to 11:15
            candles = pe[(pe['time'] >= dtime(9, 25)) & (pe['time'] <= dtime(11, 15))].copy()
            candles = candles.reset_index(drop=True)

            signal_days[d] = {
                'entry_price':  entry_price,
                'candles':      candles,
                'dte':          dte,
                'strike':       strike,
                'combo':        winning_combo,
                'nifty_open':   nifty_open,
                'nifty_925':    nifty_925,
            }
        except Exception as e:
            print(f"  [warn] {d}: {e}")
            signal_days[d] = None

    # ── Save cache ─────────────────────────────────────────────────────────
    CACHE_FILE.parent.mkdir(parents=True, exist_ok=True)
    with open(CACHE_FILE, 'wb') as f:
        pickle.dump(signal_days, f)

    valid = sum(1 for v in signal_days.values() if v is not None)
    print(f"Built trade cache: {len(signal_days)} dates, {valid} with valid signal+data.")
    print(f"  Skipped {len(skipped)} dates: "
          + ', '.join(f"{r}={sum(1 for _,x in skipped if x==r)}"
                      for r in dict.fromkeys(x for _,x in skipped)))
    print(f"  Cache saved: {CACHE_FILE}")


Fetching global market data ... done. 277 rows.
Fetching NASDAQ ... done. 280 days.
Loaded 10 bearish combos from v2_reliable_signals.csv
   1. Gap Up + Prev India DOWN + US UP + SGX UP
   2. Gap Up + Prev India DOWN + SGX UP + DAX UP
   3. Gap Up + Prev India DOWN + SGX UP + VIX Falling
   4. Gap Up + Prev India DOWN + SGX UP
   5. Gap Up + Prev India DOWN + US UP + DAX UP
   6. Gap Up + SGX UP + DAX UP + VIX Falling
   7. Gap Up + DAX UP + VIX Falling
   8. Gap Up + SGX UP + VIX Falling
   9. Gap Up + SGX UP + DAX UP
  10. Gap Up + US UP + SGX UP + DAX UP
Spot data loaded for 245 days.
Built trade cache: 189 dates, 6 with valid signal+data.
  Skipped 123 dates: Monday=53, NASDAQ_not_confirm=52, NoSpotData=5, EventDay=7, Holiday=6
  Cache saved: c:\Users\sayan\OneDrive\Desktop\Projects\03_Market_Research\market-research\gap_trading\v4\v4.6\results\trade_cache_v4_6.pkl


In [4]:
# ── Parameters dict passed to engine ──────────────────────────────────────────
params = dict(
    SL_PCT          = SL_PCT,
    TP_PCT          = TP_PCT,
    BASE_LOTS       = BASE_LOTS,
    MAX_LOTS        = MAX_LOTS,
    DTE0_LOTS       = DTE0_LOTS,
    LOT_SIZE        = LOT_SIZE,
    BEAR_N          = BEAR_N,
    ENTRY_TIME      = ENTRY_TIME,
    EXIT_TIME       = EXIT_TIME,
    INITIAL_CAPITAL = INITIAL_CAPITAL,
)

df_log = simulate_trades(signal_days, params)
print(f"Simulation complete: {len(df_log)} trades executed.")
df_log


Simulation complete: 6 trades executed.


,trade_num,date,strike,dte,entry,lots,sl_price,tp_price,exit_price,exit_reason,exit_time,pnl_pts,pnl_rs,charges_rs,capital_before,capital_after,drawdown_pct,combo,refill_rs
0,1,2024-01-04,21550,0,19.95,2,16.9575,27.93,16.9575,Stop Loss,09:28:00,-2.9925,-501.2234,52.3484,3000.0000,2498.7766,16.7074,Gap Up + Prev India DOWN + SGX UP,0.0000
1,2,2024-01-05,21650,6,82.10,2,69.7850,114.94,94.3000,11:15 exit,11:15:00,12.2000,1757.0106,72.9894,12315.0000,14072.0106,0.0000,Gap Up + SGX UP + DAX UP,9816.2234
2,3,2024-02-13,21600,2,117.60,1,99.9600,164.64,164.6400,Target Hit,09:31:00,47.0400,3459.5545,68.4455,14072.0106,17531.5651,0.0000,Gap Up + Prev India DOWN + SGX UP + DAX UP,0.0000
3,4,2024-03-27,22000,1,50.00,4,42.5000,70.00,42.5000,Stop Loss,09:27:00,-7.5000,-2323.0063,73.0063,17531.5651,15208.5588,13.2504,Gap Up + Prev India DOWN + SGX UP + DAX UP,0.0000
4,5,2024-04-18,22200,0,23.30,5,19.8050,32.62,32.6200,Target Hit,09:30:00,9.3200,3426.7532,68.2468,15208.5588,18635.3120,0.0000,Gap Up + DAX UP + VIX Falling,0.0000
5,6,2024-06-25,23550,2,92.35,2,78.4975,129.29,78.4975,Stop Loss,09:48:00,-13.8525,-2148.9072,71.0322,18635.3120,16486.4048,11.5314,Gap Up + SGX UP + DAX UP,0.0000


In [5]:
metrics = compute_metrics(df_log, params)
print_summary(df_log, metrics, params)

version_meta = dict(
    version     = VERSION,
    key_change  = KEY_CHANGE,
    data_source = DATA_SRC,
)
out = save_results(metrics, params, version_meta, HERE / 'results')
print(f"\nRun ID: {out.stem}")


  BACKTEST RESULTS  ·  2024-01-04 to 2024-06-25
  Trades          : 6
  TP hit          : 33.3%  (2 trades)
  SL hit          : 50.0%  (3 trades)
  Time exit       : 16.7%  (1 trades)
  Profitable      : 50.0%
  Avg win (Rs)    :      2,881
  Avg loss (Rs)   :     -1,658
  Total P&L (Rs)  :      3,670
  Net return      : +28.6%
  XIRR            : +70.5%
  Max drawdown    : 16.7%
  Starting cap    : Rs      3,000
  Refills         : 1x  (Rs      9,816 total)
  Total invested  : Rs     12,816
  Ending cap      : Rs     16,486
  Breakeven WR    : 27.3%  (SL=15% / TP=40%)

  Month     Trades   Wins    Win%      P&L (Rs)
  --------  ------  -----  ------  ------------
  2024-01        2      1   50.0%         1,256
  2024-02        1      1  100.0%         3,460
  2024-03        1      0    0.0%        -2,323
  2024-04        1      1  100.0%         3,427
  2024-06        1      0    0.0%        -2,149
  Results saved → c:\Users\sayan\OneDrive\Desktop\Projects\03_Market_Research\market-re